In this notebook one can:
- load a notebook's settings as a dictionary
- change it 
- save it as a new notebook 
- submit it as an array job to SLURM cluster. 

In [36]:
import sys
sys.path.append('/dls_sw/e02/software/epsic_tools')
import epsic_tools.api as ep
import pprint
import re
import subprocess
import os
import subprocess
import glob

In [42]:
starting_notebook_path = '/dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224'
starting_notebook_name = 'ACOM_template'
nb = ep.notebook_utils.NotebookHelper(starting_notebook_path, starting_notebook_name)

In [43]:
old_settings = nb.get_settings(1) # settings should be cell index 1
old_settings = old_settings.split(' ')
old_keys = [i.split('=')[0] for i in old_settings]
old_vals = [i.split('=')[1] for i in old_settings]
old_dict = dict(zip(old_keys, old_vals))
pprint.pprint(old_dict)

2026-05-27 13:47:06,551:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:31:importing jupyter code
2026-05-27 13:47:06,553:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:34:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb


{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/20260224_163652_data.hdf5',
 'save_path_name': 'ACOM_array_v2',
 'syn_probe_rad': '3',
 'syn_probe_width': '3',
 'synthetic_probe': '1',
 'v_max': '0.1',
 'v_min': '0.01'}


In [44]:
old_settings

['fill_cross=0',
 'crop_q=',
 'raw_data_path=/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/20260224_163652_data.hdf5',
 'v_min=0.01',
 'v_max=0.1',
 'synthetic_probe=1',
 'syn_probe_rad=3',
 'syn_probe_width=3',
 'probe_path=',
 'hot_pix_thresh=2',
 'load_prepared_data=0',
 'prepared_data_path=',
 'save_path_name=ACOM_array_v2']

In [45]:
# def find_hdf5_files(root_dir):
#     hdf5_files = []
#     # Loop through each subdirectory in the root directory
#     for subfolder in os.listdir(root_dir):
#         # Check if the subfolder matches the 'SPXX' pattern
#         if subfolder.startswith("Li_metal"):
#             spxx_path = os.path.join(root_dir, subfolder)
#             # Loop through each dataset subfolder inside the SPXX directory
#             for dataset_subfolder in os.listdir(spxx_path):
#                 dataset_path = os.path.join(spxx_path, dataset_subfolder)
#                 # Check for .hdf5 files in the dataset subfolder
#                 for file in os.listdir(dataset_path):
#                     if file.endswith('.hdf5'):
#                         # Append the full path of the .hdf5 file
#                         hdf5_files.append(os.path.join(dataset_path, file))
#     return hdf5_files

# Specify the root directory for the Merlin folders
merlin_root = '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight'

data_files = glob.glob(merlin_root+ '/*/*.hdf5')
#find_hdf5_files(merlin_root)
hdf5_file_paths = [path for path in data_files if 'bin' not in path]
# Output the paths
hdf5_file_paths.sort()
print(len(hdf5_file_paths))
print(*hdf5_file_paths, sep="\n")

153
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/20260224_163652_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163821/20260224_163821_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163951/20260224_163951_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164120/20260224_164120_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164351/20260224_164351_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164521/20260224_164521_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164650/20260224_164650_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164819/20260224_164819_data.hdf5
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_165050/20260224_165050_data.hd

In [46]:
hdf5_file_paths[:10]

['/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/20260224_163652_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163821/20260224_163821_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163951/20260224_163951_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164120/20260224_164120_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164351/20260224_164351_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164521/20260224_164521_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164650/20260224_164650_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_164819/20260224_164819_data.hdf5',
 '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_

In [47]:
# make some changes in new setting
# log files from the cluster jobs and the bash script will be saved here:
code_path = '/dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_logs'
concurrent_jobs = 5 #Integer number of concurrent jobs to run in the array

new_notebook_paths_list = []
for file in hdf5_file_paths:
    # update the settings
    new_setting = old_dict.copy()
    new_setting['crop_q'] = ''
    new_setting['raw_data_path'] = file
    # new_setting['save_path_name'] = 'array_cluster_processed'
    pprint.pprint(new_setting)

    save_path = os.path.join(os.path.dirname(file), new_setting['save_path_name'])
    print(save_path)
    if not os.path.exists(save_path):
        os.mkdir(save_path)

    new_notebook_path = os.path.join(save_path, 'submitted_notebook.ipynb')
    nb.set_settings(new_setting, new_notebook_path)
    print(f'new notebook path: {new_notebook_path}')
    new_notebook_paths_list.append(new_notebook_path)

note_book_path_file = os.path.join(code_path, 'notebook_list.txt')
with open (note_book_path_file, 'w') as f:
    f.write(
        '\n'.join(new_notebook_paths_list)
    )


bash_script_path = os.path.join(code_path, 'cluster_submit.sh')
with open (bash_script_path, 'w') as f:
    f.write('''#!/usr/bin/env bash
#SBATCH --partition cs05r
#SBATCH --job-name epsic_notebook
#SBATCH --time 06:00:00
#SBATCH --nodes 1
#SBATCH --gpus-per-node=1
#SBATCH --tasks-per-node 1
#SBATCH --mem 40G
'''
f"#SBATCH --array=0-{len(new_notebook_paths_list)-1}%{int(concurrent_jobs)}\n"
f"#SBATCH --error={code_path}{os.sep}logs{os.sep}error_%j.out\n"
f"#SBATCH --output={code_path}{os.sep}logs{os.sep}output_%j.out\n"
f"module load python/epsic3.10\n"
f"mapfile -t paths_array < {note_book_path_file}\n"
'''
echo ${paths_array[$SLURM_ARRAY_TASK_ID]}
jupyter nbconvert --to notebook --inplace --ClearMetadataPreprocessor.enabled=True ${paths_array[$SLURM_ARRAY_TASK_ID]}
jupyter nbconvert --to notebook --execute ${paths_array[$SLURM_ARRAY_TASK_ID]}

'''
           )
        
sshProcess = subprocess.Popen(['ssh',
                               '-tt',
                               'wilson'],
                               stdin=subprocess.PIPE, 
                               stdout = subprocess.PIPE,
                               universal_newlines=True,
                               bufsize=0)
sshProcess.stdin.write("ls .\n")
sshProcess.stdin.write("echo END\n")
sshProcess.stdin.write(f"sbatch {bash_script_path}\n")
sshProcess.stdin.write("uptime\n")
sshProcess.stdin.write("logout\n")
sshProcess.stdin.close()


for line in sshProcess.stdout:
    if line == "END\n":
        break
    print(line,end="")

#to catch the lines up to logout
for line in  sshProcess.stdout: 
    print(line,end="")
            


2026-05-27 13:47:11,840:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:11,842:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:11,870:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:11,870:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:11,898:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:11,899:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/scie

{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/20260224_163652_data.hdf5',
 'save_path_name': 'ACOM_array_v2',
 'syn_probe_rad': '3',
 'syn_probe_width': '3',
 'synthetic_probe': '1',
 'v_max': '0.1',
 'v_min': '0.01'}
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/ACOM_array_v2
new notebook path: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163652/ACOM_array_v2/submitted_notebook.ipynb
{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_163821/20260224_163821_data.hdf5',
 'save_path_name': 'ACOM_array_v2',
 'syn_probe_rad': '3',
 'syn_probe_width': '3',


2026-05-27 13:47:12,018:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:12,018:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:12,032:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:12,032:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:12,058:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:12,059:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/scie

new notebook path: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_165220/ACOM_array_v2/submitted_notebook.ipynb
{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_165349/20260224_165349_data.hdf5',
 'save_path_name': 'ACOM_array_v2',
 'syn_probe_rad': '3',
 'syn_probe_width': '3',
 'synthetic_probe': '1',
 'v_max': '0.1',
 'v_min': '0.01'}
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_165349/ACOM_array_v2
new notebook path: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_165349/ACOM_array_v2/submitted_notebook.ipynb
{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_2

2026-05-27 13:47:12,237:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:12,238:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:12,254:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:12,255:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:12,269:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:12,270:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/scie

...

2026-05-27 13:47:14,363:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:14,380:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:14,381:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:14,396:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:14,397:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ip

new notebook path: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_203027/ACOM_array_v2/submitted_notebook.ipynb
{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_203157/20260224_203157_data.hdf5',
 'save_path_name': 'ACOM_array_v2',
 'syn_probe_rad': '3',
 'syn_probe_width': '3',
 'synthetic_probe': '1',
 'v_max': '0.1',
 'v_min': '0.01'}
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_203157/ACOM_array_v2
new notebook path: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_203157/ACOM_array_v2/submitted_notebook.ipynb
{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_2

2026-05-27 13:47:14,578:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:14,596:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:14,597:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ipynb
2026-05-27 13:47:14,613:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:76:importing jupyter code
2026-05-27 13:47:14,614:/dls_sw/e02/software/epsic_tools/epsic_tools/toolbox/notebook_utils.py:79:reading notebook from /dls/science/groups/e02/Mohsen/code/jupyterhub_active/Scan_GUI_Automator_workflow/Pt_NP_autoSTEM_session_20260224/ACOM_template.ip

new notebook path: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_204955/ACOM_array_v2/submitted_notebook.ipynb
{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_205125/20260224_205125_data.hdf5',
 'save_path_name': 'ACOM_array_v2',
 'syn_probe_rad': '3',
 'syn_probe_width': '3',
 'synthetic_probe': '1',
 'v_max': '0.1',
 'v_min': '0.01'}
/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_205125/ACOM_array_v2
new notebook path: /dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight/20260224_205125/ACOM_array_v2/submitted_notebook.ipynb
{'crop_q': '',
 'fill_cross': '0',
 'hot_pix_thresh': '2',
 'load_prepared_data': '0',
 'prepared_data_path': '',
 'probe_path': '',
 'raw_data_path': '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_2

Connection to wilson closed.
